## Neural Networks and LLMs

In [1]:
print('hi')

hi


In [2]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR


In [5]:
LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [6]:
username = "SeanSunny"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


# a vanilla Neural Network

In [5]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [6]:
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the CountVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [7]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [8]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [9]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [ ]:
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [1/5], Train Loss: 9053.730, Val Loss: 11958.276


  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [2/5], Train Loss: 3753.683, Val Loss: 11400.784


  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [3/5], Train Loss: 4887.390, Val Loss: 11127.543


  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [4/5], Train Loss: 8835.807, Val Loss: 11463.605


  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [5/5], Train Loss: 6292.738, Val Loss: 11720.070


In [11]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [12]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$97 $114 $27 $20 $21 $146 $61 $31 $15 $154 $24 $250 $25 $21 $10 $13 $4 $10 $30 $32 $19 $13 $2 $81 $174 $280 $228 $31 $69 $48 $106 $74 $23 $14 $29 $124 $24 $15 $117 $46 $134 $13 $38 $39 $43 $37 $28 $12 $80 $10 $4 $43 $175 $46 $53 $70 $24 $75 $7 $40 $99 $19 $21 $36 $224 $75 $5 $384 $13 $57 $8 $15 $107 $134 $9 $40 $189 $9 $21 $49 $52 $48 $44 $38 $16 $196 $43 $96 $46 $148 $23 $9 $9 $8 $30 $37 $6 $4 $1 $159 $9 $42 $38 $76 $19 $226 $57 $292 $3 $141 $35 $54 $97 $27 $26 $63 $59 $97 $7 $117 $23 $105 $41 $20 $91 $62 $19 $41 $107 $27 $80 $33 $72 $15 $50 $21 $54 $51 $5 $37 $3 $152 $22 $59 $14 $32 $17 $248 $12 $2 $21 $85 $12 $53 $14 $84 $53 $9 $81 $19 $134 $5 $1 $22 $145 $12 $265 $32 $14 $44 $47 $10 $250 $36 $21 $18 $36 $16 $26 $225 $157 $11 $13 $128 $113 $25 $50 $24 $30 $30 $37 $26 $7 $29 $11 $3 $181 $28 $1 $17 

## GPT

In [7]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [8]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [9]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [8]:
# The function for gpt-4.1-nano

def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [9]:
gpt_4__1_nano(test[0])

'$180'

In [10]:
test[0].price

219.0

In [11]:
evaluate(gpt_4__1_nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$19 $134 $30 $20 $20 $80 $6 $65 $11 $520 $363 $20 $30 $16 $1 $8 $41 $5 $40 $31 $54 $26 $65 $75 $182 $254 $205 $5 $151 $64 $60 $15 $10 $60 $5 $169 $90 $31 $6 $13 $165 $55 $25 $105 $120 $0 $12 $13 $75 $52 $23 $105 $125 $10 $247 $14 $8 $20 $52 $3 $86 $28 $41 $40 $21 $9 $90 $295 $25 $44 $16 $8 $130 $1 $25 $21 $76 $0 $8 $0 $30 $3 $15 $74 $11 $10 $32 $44 $0 $1 $13 $15 $5 $19 $2 $78 $1 $7 $120 $325 $20 $3 $7 $19 $51 $32 $10 $350 $1 $49 $0 $286 $49 $78 $24 $180 $5 $5 $44 $47 $24 $511 $50 $16 $50 $10 $5 $151 $59 $89 $129 $13 $35 $5 $55 $0 $55 $10 $78 $62 $16 $100 $70 $12 $114 $43 $15 $340 $15 $18 $3 $144 $2 $10 $1 $129 $101 $41 $30 $5 $211 $17 $7 $3 $140 $7 $752 $25 $5 $5 $0 $2 $80 $8 $32 $101 $3 $57 $56 $23 $546 $35 $150 $1 $100 $3 $73 $7 $20 $2 $15 $99 $15 $11 $40 $70 $10 $30 $21 $1 

In [12]:
# The function for gpt-5.1

def gpt_5__1(item):
    response = completion(model="openai/gpt-5.1", messages=messages_for(item))
    return response.choices[0].message.content

In [13]:
evaluate(gpt_5__1, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$10 $114 $5 $10 $30 $120 $54 $80 $7 $100 $13 $70 $5 $14 $39 $8 $11 $15 $50 $39 $36 $14 $5 $25 $52 $204 $16 $1 $81 $61 $20 $20 $30 $57 $15 $169 $60 $34 $34 $13 $160 $40 $11 $65 $20 $5 $7 $1 $60 $38 $26 $111 $255 $0 $27 $6 $6 $50 $108 $6 $116 $48 $56 $50 $21 $0 $50 $295 $20 $54 $15 $2 $30 $1 $25 $17 $36 $2 $2 $4 $10 $4 $13 $74 $7 $25 $63 $46 $30 $21 $8 $15 $5 $5 $1 $78 $3 $72 $10 $255 $40 $13 $3 $50 $49 $102 $16 $340 $6 $69 $20 $306 $9 $8 $54 $30 $5 $0 $24 $147 $14 $9 $10 $6 $10 $15 $5 $41 $19 $69 $139 $8 $1 $0 $85 $2 $55 $10 $52 $12 $31 $349 $5 $1 $4 $28 $10 $80 $85 $8 $2 $114 $22 $50 $0 $19 $46 $36 $30 $10 $89 $18 $2 $1 $91 $7 $352 $20 $10 $4 $5 $3 $170 $8 $56 $81 $4 $27 $26 $2 $4 $25 $220 $29 $20 $8 $63 $12 $10 $3 $5 $19 $10 $121 $50 $50 $20 $40 $11 $6 